In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "credit_default_raw.xls"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "credit_default_clean.csv"

# Load raw data
df = pd.read_excel(RAW_DATA_PATH, header=1)

# Create a copy for cleaning
df_clean = df.copy()

print("Raw dataset shape:", df.shape)
print("Cleaning dataset shape:", df_clean.shape)

df_clean.head()

Raw dataset shape: (30000, 25)
Cleaning dataset shape: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [9]:
# Rename target column

df_clean = df_clean.rename(
    columns={
        "default payment next month": "TARGET"
    }
)

print(df_clean.columns.tolist())

['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'TARGET']


In [10]:
# Clean undocumented categorical values

# EDUCATION:
# 1 = Graduate school
# 2 = University
# 3 = High school
# 4 = Others
# Undocumented values 0, 5, 6 are grouped into Others.

df_clean["EDUCATION"] = df_clean["EDUCATION"].replace({
    0: 4,
    5: 4,
    6: 4
})

# MARRIAGE:
# 1 = Married
# 2 = Single
# 3 = Others
# Undocumented value 0 is grouped into Others.

df_clean["MARRIAGE"] = df_clean["MARRIAGE"].replace({
    0: 3
})
print("EDUCATION after cleaning:")
print(df_clean["EDUCATION"].value_counts().sort_index())

print("\nMARRIAGE after cleaning:")
print(df_clean["MARRIAGE"].value_counts().sort_index())

EDUCATION after cleaning:
EDUCATION
1    10585
2    14030
3     4917
4      468
Name: count, dtype: int64

MARRIAGE after cleaning:
MARRIAGE
1    13659
2    15964
3      377
Name: count, dtype: int64


In [11]:
payment_status_cols = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

print("Payment status validation:")

for col in payment_status_cols:
    print(
        f"{col}: "
        f"min={df_clean[col].min()}, "
        f"max={df_clean[col].max()}, "
        f"unique={sorted(df_clean[col].unique())}"
    )

Payment status validation:
PAY_0: min=-2, max=8, unique=[np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
PAY_2: min=-2, max=8, unique=[np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
PAY_3: min=-2, max=8, unique=[np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
PAY_4: min=-2, max=8, unique=[np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
PAY_5: min=-2, max=8, unique=[np.int64(-2), np.int64(-1), np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
PAY_6: min=-2, max=8, unique=[np.int64(-2), np.int64(-1), np.int64(0), np.int64(2), np.int64(3), np.int64(4), n

In [12]:
print("===== FINAL CLEANING CHECK =====")

print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])

print("\nMissing values:", df_clean.isnull().sum().sum())
print("Duplicate rows:", df_clean.duplicated().sum())
print("Duplicate IDs:", df_clean["ID"].duplicated().sum())

print("\nTARGET values:")
print(df_clean["TARGET"].value_counts().sort_index())

print("\nEDUCATION values:")
print(sorted(df_clean["EDUCATION"].unique()))

print("\nMARRIAGE values:")
print(sorted(df_clean["MARRIAGE"].unique()))

print("\nSEX values:")
print(sorted(df_clean["SEX"].unique()))

===== FINAL CLEANING CHECK =====
Rows: 30000
Columns: 25

Missing values: 0
Duplicate rows: 0
Duplicate IDs: 0

TARGET values:
TARGET
0    23364
1     6636
Name: count, dtype: int64

EDUCATION values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

MARRIAGE values:
[np.int64(1), np.int64(2), np.int64(3)]

SEX values:
[np.int64(1), np.int64(2)]


In [13]:
# Save cleaned dataset

PROCESSED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "credit_default_clean.csv"
)

df_clean.to_csv(
    PROCESSED_DATA_PATH,
    index=False
)

print("Saved cleaned dataset to:")
print(PROCESSED_DATA_PATH)

print("\nShape:", df_clean.shape)

Saved cleaned dataset to:
h:\Credit_Risk_Modeling\data\processed\credit_default_clean.csv

Shape: (30000, 25)


## Data Cleaning Summary

- Raw dataset contained 30,000 rows and 25 columns.
- No missing values were detected.
- No duplicated rows or duplicated customer IDs were found.
- Renamed `default payment next month` to `TARGET`.
- Standardized undocumented `EDUCATION` categories:
  - 0, 5, 6 → 4 (Others)
- Standardized undocumented `MARRIAGE` category:
  - 0 → 3 (Others)
- Original payment status variables `PAY_0` to `PAY_6` were preserved.
- Negative and zero payment-status codes were not removed because they may contain useful behavioral information.
- Negative `BILL_AMT` values were preserved for further analysis.
- Large bill/payment amounts were not automatically removed and will be investigated during EDA.
- `ID` was preserved as a customer identifier but will not be used as a model feature.
- Final cleaned dataset:
  - 30,000 rows
  - 25 columns
  - 0 missing values